In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Try to get the dataframe from memory or load it
if 'processed_all_train_data' in locals():
    df_check = processed_all_train_data
elif os.path.exists('./data/train_data.csv'):
    print("Loading data from disk...")
    df_check = pl.read_csv('./data/train_data.csv')
else:
    df_check = None
    print("Data not found. Please ensure the training data processing cells have been run.")

if df_check is not None:
    # Display summary statistics
    print("Target (Average Delta Run Exp) Statistics:")
    print(df_check['target'].describe())

    # Plot distribution
    plt.figure(figsize=(10, 6))
    # Sample data for plotting if it's too large
    plot_data = df_check['target'].sample(n=100000, seed=42) if df_check.height > 100000 else df_check['target']

    sns.histplot(plot_data.to_numpy(), bins=50, kde=True)
    plt.title('Distribution of Target Values (Avg Delta Run Exp)')
    plt.xlabel('Target Value (Runs)')
    plt.ylabel('Frequency')
    plt.grid(alpha=0.3)
    plt.show()

    # Display some examples of high/low values
    print("\nExamples of distinct target values:")
    print(df_check.select(['des_new', 'count', 'target']).unique().sort('target').head(5))
    print(df_check.select(['des_new', 'count', 'target']).unique().sort('target', descending=True).head(5))

In [ ]:
!cp -r "/content/drive/MyDrive/stuff_model/data/" "/content/"

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import joblib


In [ ]:
# Define a dictionary to group outcomes together
des_dict = {
    'ball': 'ball',
    'hit_into_play': 'hit_into_play',
    'called_strike': 'called_strike',
    'foul': 'foul',
    'swinging_strike': 'swinging_strike',
    'blocked_ball': 'ball',
    'swinging_strike_blocked': 'swinging_strike',
    'foul_tip': 'swinging_strike',
    'foul_bunt': 'foul',
    'hit_by_pitch': 'hit_by_pitch',
    'pitchout': 'ball',
    'missed_bunt': 'swinging_strike',
    'bunt_foul_tip': 'swinging_strike',
    'foul_pitchout': 'foul',
}

# Define a dictionary to group events together
ev_dict = {
    'single': 'single',
    'walk': 'walk',
    'strikeout': 'strikeout',
    'field_out': 'field_out',
    'force_out': 'field_out',
    'double': 'double',
    'hit_by_pitch': 'hit_by_pitch',
    'home_run': 'home_run',
    'grounded_into_double_play': 'field_out',
    'fielders_choice_out': 'field_out',
    'fielders_choice': 'field_out',
    'field_error': 'field_out',
    'double_play': 'field_out',
    'sac_fly': 'field_out',
    'triple': 'triple',
    'sac_bunt': 'field_out',
    'wild_pitch': 'wild_pitch',
    'sac_fly_double_play': 'field_out',
    'triple_play': 'field_out',
    'other_out': 'field_out',
    'sac_bunt_double_play': 'field_out',
}

In [ ]:
swing_descriptions = ['foul_bunt', 'foul', 'hit_into_play', 'swinging_strike', 'foul_tip', 'swinging_strike_blocked', 'missed_bunt', 'bunt_foul_tip', 'foul_pitchout']

In [ ]:

# Pitch types that are incompatible with the model's spin/movement feature assumptions
EXCLUDED_PITCH_TYPES = [
    'KN',  # Knuckleball — ~0 spin rate, spin axis meaningless
    'EP',  # Eephus — 40-55 mph, extreme velocity outlier
    'PO',  # Pitchout — not a real pitch attempt
    'SC',  # Screwball — essentially extinct, only 235 total across 5 years
    'UN',  # Unknown classification — no meaningful features
    'CS',  # Slow curve — rare, extreme outlier in break profile
]

def df_clean(df):
    """
    Clean and transform baseball data using Polars.
    NOTE: Does NOT calculate target values - those must be calculated separately
    from training data only to avoid data leakage.

    Args:
        df: Polars DataFrame with baseball pitch data

    Returns:
        Cleaned and transformed Polars DataFrame (without target column)
    """

    # Filter out null and incompatible pitch types
    df = df.filter(
        pl.col('pitch_type').is_not_null() &
        ~pl.col('pitch_type').is_in(EXCLUDED_PITCH_TYPES)
    )

    # Filter out physically unrealistic values (sensor artifacts / tracking errors)
    # Null values are preserved — they are handled downstream
    df = df.filter(
        (pl.col('release_speed').is_null()      | pl.col('release_speed').is_between(40, 110)) &
        (pl.col('release_spin_rate').is_null()  | pl.col('release_spin_rate').is_between(500, 4000)) &
        (pl.col('release_extension').is_null()  | pl.col('release_extension').is_between(0, 9))
    )

    # Create new column with grouped descriptions
    df = df.with_columns([
        pl.col("description").replace_strict(des_dict, default = None).alias("des_new")
    ])

    # Create new column with grouped events for hit_into_play cases
    df = df.with_columns([
        pl.when(pl.col("des_new") == "hit_into_play")
        .then(pl.col("events").replace_strict(ev_dict, default = None))
        .otherwise(None)
        .alias("ev_new")
    ])

    # Replace des_new with ev_new for hit_into_play cases
    df = df.with_columns([
        pl.when(pl.col("des_new") == "hit_into_play")
        .then(pl.col("ev_new"))
        .otherwise(pl.col("des_new"))
        .alias("des_new")
    ]).drop("ev_new")

    # Filter out null des_new values
    df = df.filter(pl.col("des_new").is_not_null())

    # Filter rare cases (strikes <= 2 and balls <= 3)
    df = df.filter(
        (pl.col("strikes") <= 2) &
        (pl.col("balls") <= 3)
    )

    # Create count column
    df = df.with_columns([
        (pl.col("balls").cast(pl.Utf8) + "-" + pl.col("strikes").cast(pl.Utf8)).alias("count")
    ])

    # Add binary column for swings
    df = df.with_columns([
        pl.col("description").is_in(swing_descriptions).alias("swing")
    ])

    # Cast the year column to integer type
    df = df.with_columns(
        pl.col('game_year').cast(pl.Int64)
    )

    return df


In [ ]:
def calculate_target_lookup(train_df):
    """
    Calculate average run expectancy values from TRAINING data only.
    These values will be applied to both train and test sets.
    """
    target_lookup = train_df.group_by(["des_new", "count", "p_throws", "stand"]).agg([
        pl.col("delta_run_exp").mean().alias("target")
    ])
    return target_lookup


def apply_target_values(df, target_lookup):
    """Apply pre-calculated target values to a DataFrame."""
    df = df.join(
        target_lookup,
        on=["des_new", "count", "p_throws", "stand"],
        how="left"
    )
    return df


def compute_vaa(df: pl.DataFrame) -> pl.DataFrame:
    """
    Compute Vertical Approach Angle (VAA) for each pitch using Statcast kinematics.

    VAA is the angle of the ball's trajectory as it crosses home plate (degrees).
    Negative = descending, closer to 0 = flatter trajectory (harder to hit for fastballs).

    LHP-agnostic: VAA is purely vertical and unaffected by handedness mirroring.
    Requires: vy0, vz0, ay, az columns from raw Statcast.
    """
    required = ['vy0', 'vz0', 'ay', 'az']
    if not all(c in df.columns for c in required):
        print("Warning: VAA computation skipped — missing kinematic columns (vy0, vz0, ay, az)")
        return df

    Y_PLATE  = 17 / 12   # front of home plate (feet)
    Y_INIT   = 50.0      # Statcast initial conditions measured at y=50 ft
    RAD_TO_DEG = 180.0 / np.pi

    # Solve for time to reach home plate:
    # Y_PLATE = Y_INIT + vy0*t + 0.5*ay*t²
    # => t = (-vy0 - sqrt(vy0² - 2*ay*(Y_INIT - Y_PLATE))) / ay
    df = (
        df
        .with_columns([
            (pl.col('vy0')**2 - 2 * pl.col('ay') * (Y_INIT - Y_PLATE)).alias('_vaa_disc')
        ])
        .with_columns([
            ((-pl.col('vy0') - pl.col('_vaa_disc').sqrt()) / pl.col('ay')).alias('_vaa_t')
        ])
        .with_columns([
            (pl.col('vy0') + pl.col('ay') * pl.col('_vaa_t')).alias('_vy_f'),
            (pl.col('vz0') + pl.col('az') * pl.col('_vaa_t')).alias('_vz_f'),
        ])
        .with_columns([
            ((pl.col('_vz_f') / pl.col('_vy_f').abs()).arctan() * RAD_TO_DEG).alias('VAA')
        ])
        .drop(['_vaa_disc', '_vaa_t', '_vy_f', '_vz_f'])
    )
    return df


def calculate_pitcher_stats_lookup(train_df):
    """
    Calculate pitcher statistics from TRAINING data only.
    Includes fastball averages for speed, movement, and VAA (if available).
    FA (generic fastball) is treated identically to FF.
    """
    fastball_pitch_types = ['FF', 'SI', 'FC', 'FA']  # FA = generic fastball tag

    df_filtered = train_df.filter(pl.col('pitch_type').is_in(fastball_pitch_types))

    # Aggregate columns that always exist
    agg_exprs = [
        pl.col('release_speed').mean().alias('avg_fastball_speed'),
        pl.col('az').mean().alias('avg_fastball_az'),
        pl.col('ax').mean().alias('avg_fastball_ax'),
        pl.len().alias('pitch_count'),
    ]
    if 'VAA' in train_df.columns:
        agg_exprs.append(pl.col('VAA').mean().alias('avg_fastball_vaa'))

    df_agg = df_filtered.group_by(['pitcher', 'game_year', 'pitch_type']).agg(agg_exprs)

    # Get the most-used fastball type per pitcher-year
    df_agg = df_agg.sort('pitch_count', descending=True)
    fastball_stats = df_agg.unique(subset=['pitcher', 'game_year'], keep='first')

    # Fallback: fastest pitch for pitchers without a standard fastball
    fastest_pitch_info = (
        train_df.sort('release_speed', descending=True)
        .group_by('pitcher')
        .agg([pl.col('pitch_type').first().alias('fastest_pitch_type')])
    )

    fallback_agg = [
        pl.col('release_speed').mean().alias('avg_fastest_pitch_speed'),
        pl.col('az').mean().alias('avg_fastest_pitch_az'),
        pl.col('ax').mean().alias('avg_fastest_pitch_ax'),
    ]
    if 'VAA' in train_df.columns:
        fallback_agg.append(pl.col('VAA').mean().alias('avg_fastest_pitch_vaa'))

    fastest_pitch_stats = (
        train_df.join(fastest_pitch_info, on='pitcher')
        .filter(pl.col('pitch_type') == pl.col('fastest_pitch_type'))
        .group_by(['pitcher', 'game_year'])
        .agg(fallback_agg)
    )

    # Merge: fastball stats where available, fallback otherwise
    join_cols = ['avg_fastball_speed', 'avg_fastball_az', 'avg_fastball_ax']
    if 'VAA' in train_df.columns:
        join_cols.append('avg_fastball_vaa')

    coalesce_exprs = [
        pl.coalesce(['avg_fastball_speed', 'avg_fastest_pitch_speed']).alias('avg_fastball_speed'),
        pl.coalesce(['avg_fastball_az',    'avg_fastest_pitch_az'   ]).alias('avg_fastball_az'),
        pl.coalesce(['avg_fastball_ax',    'avg_fastest_pitch_ax'   ]).alias('avg_fastball_ax'),
    ]
    if 'VAA' in train_df.columns:
        coalesce_exprs.append(
            pl.coalesce(['avg_fastball_vaa', 'avg_fastest_pitch_vaa']).alias('avg_fastball_vaa')
        )

    select_cols = ['pitcher', 'game_year', 'avg_fastball_speed', 'avg_fastball_az', 'avg_fastball_ax']
    if 'VAA' in train_df.columns:
        select_cols.append('avg_fastball_vaa')

    pitcher_stats = (
        fastest_pitch_stats
        .join(
            fastball_stats.select(['pitcher', 'game_year'] + join_cols),
            on=['pitcher', 'game_year'],
            how='left'
        )
        .with_columns(coalesce_exprs)
        .select(select_cols)
    )
    return pitcher_stats


def calculate_magnus_model(train_df: pl.DataFrame):
    """
    Train simple linear regressions to predict pfx_x and pfx_z from pure
    Magnus/physics features. Residuals become the SSW proxy.

    Uses LinearRegression intentionally (not gradient boosting) to avoid
    absorbing SSW signal into the Magnus prediction.

    Requires: release_speed, release_spin_rate, spin_axis,
              release_extension, pfx_x, pfx_z columns.

    Returns: (magnus_model_x, magnus_model_z) sklearn LinearRegression instances
    """
    required = ['release_speed', 'release_spin_rate', 'spin_axis',
                'release_extension', 'pfx_x', 'pfx_z']
    missing = [c for c in required if c not in train_df.columns]
    if missing:
        raise ValueError(f"Magnus model training failed — missing columns: {missing}")

    # sin/cos encoding for spin_axis (relationship is sinusoidal; raw degrees break at wraparound)
    df_m = train_df.with_columns([
        (pl.col('spin_axis') * np.pi / 180).sin().alias('_spin_sin'),
        (pl.col('spin_axis') * np.pi / 180).cos().alias('_spin_cos'),
    ])

    magnus_cols = ['release_speed', 'release_spin_rate',
                   '_spin_sin', '_spin_cos', 'release_extension']

    X_pd = df_m.select(magnus_cols + ['pfx_x', 'pfx_z']).to_pandas().dropna()

    X  = X_pd[magnus_cols].values
    y_x = X_pd['pfx_x'].values
    y_z = X_pd['pfx_z'].values

    magnus_model_x = LinearRegression().fit(X, y_x)
    magnus_model_z = LinearRegression().fit(X, y_z)

    print(f"Magnus model trained on {len(X):,} pitches")
    print(f"  pfx_x R²: {magnus_model_x.score(X, y_x):.4f}")
    print(f"  pfx_z R²: {magnus_model_z.score(X, y_z):.4f}")

    return magnus_model_x, magnus_model_z


In [ ]:
def feature_engineering(df: pl.DataFrame, pitcher_stats_lookup: pl.DataFrame,
                         magnus_model_x=None, magnus_model_z=None) -> pl.DataFrame:
    """
    Apply feature engineering to the dataset.

    Args:
        df: Cleaned DataFrame with VAA pre-computed via compute_vaa()
        pitcher_stats_lookup: From calculate_pitcher_stats_lookup() — includes avg_fastball_vaa
        magnus_model_x: Trained LinearRegression for pfx_x SSW proxy (optional)
        magnus_model_z: Trained LinearRegression for pfx_z SSW proxy (optional)

    Returns:
        DataFrame with all engineered features. Raw kinematic columns are dropped.
    """
    Y_PLATE    = 17 / 12
    Y_INIT     = 50.0
    RAD_TO_DEG = 180.0 / np.pi

    # ── Handedness mirroring (put all pitchers in right-hand frame of reference) ──
    df = df.with_columns(
        pl.when(pl.col('p_throws') == "L")
        .then(-pl.col('ax'))
        .otherwise(pl.col('ax'))
        .alias('ax')
    )
    df = df.with_columns(
        pl.when(pl.col('p_throws') == "L")
        .then(-pl.col('release_pos_x'))
        .otherwise(pl.col('release_pos_x'))
        .alias('release_pos_x')
    )
    # Mirror vx0 and pfx_x for LHP (HAA and SSW are horizontal — same convention as ax)
    if 'vx0' in df.columns:
        df = df.with_columns(
            pl.when(pl.col('p_throws') == "L")
            .then(-pl.col('vx0'))
            .otherwise(pl.col('vx0'))
            .alias('vx0')
        )
    if 'pfx_x' in df.columns:
        df = df.with_columns(
            pl.when(pl.col('p_throws') == "L")
            .then(-pl.col('pfx_x'))
            .otherwise(pl.col('pfx_x'))
            .alias('pfx_x')
        )

    # ── HAA (Horizontal Approach Angle) ──
    # Computed after LHP mirroring so it's in the right-hand pitcher frame.
    if all(c in df.columns for c in ['vx0', 'vy0', 'ay']):
        df = (
            df
            .with_columns([
                (pl.col('vy0')**2 - 2 * pl.col('ay') * (Y_INIT - Y_PLATE)).alias('_haa_disc')
            ])
            .with_columns([
                ((-pl.col('vy0') - pl.col('_haa_disc').sqrt()) / pl.col('ay')).alias('_haa_t')
            ])
            .with_columns([
                (pl.col('vx0') + pl.col('ax') * pl.col('_haa_t')).alias('_vx_f'),
                (pl.col('vy0') + pl.col('ay') * pl.col('_haa_t')).alias('_vy_f'),
            ])
            .with_columns([
                ((pl.col('_vx_f') / pl.col('_vy_f').abs()).arctan() * RAD_TO_DEG).alias('HAA')
            ])
            .drop(['_haa_disc', '_haa_t', '_vx_f', '_vy_f'])
        )

    # Drop raw kinematic columns — no longer needed after VAA/HAA are computed
    drop_kinematics = [c for c in ['vx0', 'vy0', 'vz0', 'ay'] if c in df.columns]
    if drop_kinematics:
        df = df.drop(drop_kinematics)

    # ── Pitcher stat differentials ──
    df = df.join(pitcher_stats_lookup, on=['pitcher', 'game_year'], how='left')
    df = df.with_columns([
        (pl.col('release_speed') - pl.col('avg_fastball_speed')).alias('speed_diff'),
        (pl.col('az')            - pl.col('avg_fastball_az')   ).alias('az_diff'),
        (pl.col('ax')            - pl.col('avg_fastball_ax')   ).alias('ax_diff'),
    ])

    # VAA differential — VAA relative to pitcher's fastball (tunneling deception)
    if 'VAA' in df.columns and 'avg_fastball_vaa' in df.columns:
        df = df.with_columns([
            (pl.col('VAA') - pl.col('avg_fastball_vaa')).alias('vaa_diff')
        ])

    # ── SSW proxy features ──
    # Residual between actual movement (pfx_x/z) and Magnus-predicted movement.
    ssw_required = ['release_speed', 'release_spin_rate', 'spin_axis',
                    'release_extension', 'pfx_x', 'pfx_z']
    if magnus_model_x is not None and magnus_model_z is not None and \
       all(c in df.columns for c in ssw_required):

        df = df.with_columns([
            (pl.col('spin_axis') * np.pi / 180).sin().alias('_spin_sin'),
            (pl.col('spin_axis') * np.pi / 180).cos().alias('_spin_cos'),
        ])
        magnus_cols = ['release_speed', 'release_spin_rate',
                       '_spin_sin', '_spin_cos', 'release_extension']

        X_pd = df.select(magnus_cols).to_pandas()
        valid_rows = ~X_pd.isnull().any(axis=1)

        pred_pfx_x = np.full(len(df), np.nan)
        pred_pfx_z = np.full(len(df), np.nan)
        if valid_rows.sum() > 0:
            pred_pfx_x[valid_rows] = magnus_model_x.predict(X_pd[valid_rows].values)
            pred_pfx_z[valid_rows] = magnus_model_z.predict(X_pd[valid_rows].values)

        df = (
            df
            .with_columns([
                pl.Series('_pred_pfx_x', pred_pfx_x, dtype=pl.Float64),
                pl.Series('_pred_pfx_z', pred_pfx_z, dtype=pl.Float64),
            ])
            .with_columns([
                (pl.col('pfx_x') - pl.col('_pred_pfx_x')).alias('ssw_x'),
                (pl.col('pfx_z') - pl.col('_pred_pfx_z')).alias('ssw_z'),
            ])
            .with_columns([
                ((pl.col('ssw_x')**2 + pl.col('ssw_z')**2).sqrt()).alias('ssw_magnitude')
            ])
            .drop(['_spin_sin', '_spin_cos', '_pred_pfx_x', '_pred_pfx_z', 'pfx_x', 'pfx_z'])
        )

    return df


In [ ]:
def validate_data(df, dataset_name="dataset"):
    """
    Validate that the dataset has expected structure and values.

    Args:
        df: DataFrame to validate
        dataset_name: Name of dataset for error messages

    Raises:
        ValueError: If validation fails
    """
    required_cols = ['description', 'events', 'strikes', 'balls', 'game_year',
                     'p_throws', 'stand', 'delta_run_exp', 'pitch_type',
                     'ax', 'az', 'release_speed', 'release_pos_x', 'pitcher']

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{dataset_name}: Missing required columns: {missing_cols}")

    # Check for valid strike/ball counts
    invalid_strikes = df.filter((pl.col('strikes') < 0) | (pl.col('strikes') > 3)).height
    invalid_balls = df.filter((pl.col('balls') < 0) | (pl.col('balls') > 4)).height

    if invalid_strikes > 0:
        print(f"Warning: {dataset_name} has {invalid_strikes} rows with invalid strike counts")
    if invalid_balls > 0:
        print(f"Warning: {dataset_name} has {invalid_balls} rows with invalid ball counts")

    # Check for duplicate pitches
    id_cols = ['game_pk', 'at_bat_number', 'pitch_number']
    if all(c in df.columns for c in id_cols):
        duplicate_count = df.height - df.unique(subset=id_cols).height
        if duplicate_count > 0:
            print(f"Warning: {dataset_name} has {duplicate_count} duplicate pitches")

    # Report null counts in critical columns
    for col in ['pitch_type', 'release_speed', 'ax', 'az', 'delta_run_exp']:
        null_count = df.filter(pl.col(col).is_null()).height
        if null_count > 0:
            print(f"Info: {dataset_name} has {null_count} null values in '{col}' ({null_count/df.height*100:.2f}%)")

    print(f"{dataset_name} validation complete. Shape: {df.shape}")
    return True


In [ ]:
# Load data using Polars
data_2021 = pl.read_csv('./data/2021_data.csv')
data_2022 = pl.read_csv('./data/2022_data.csv')
data_2023 = pl.read_csv('./data/2023_data.csv')
data_2024 = pl.read_csv('./data/2024_data.csv')

In [ ]:
# Concatenate all training years
all_train_data = pl.concat([data_2021, data_2022, data_2023, data_2024], how = 'vertical_relaxed')

# Validate raw data
validate_data(all_train_data, "Raw training data")

Info: Raw training data has 10567 null values in 'pitch_type' (0.37%)
Info: Raw training data has 10347 null values in 'release_speed' (0.36%)
Info: Raw training data has 10351 null values in 'ax' (0.36%)
Info: Raw training data has 10351 null values in 'az' (0.36%)
Info: Raw training data has 9279 null values in 'delta_run_exp' (0.33%)
Raw training data validation complete. Shape: (2854422, 118)


True

In [ ]:
# Columns to keep in the final output — drops the ~100 unused raw Statcast columns
FINAL_COLS = [
    # Identity & metadata
    'pitcher', 'player_name', 'pitch_type', 'game_year', 'game_date', 'game_pk',
    # Original model features (13)
    'release_speed', 'release_spin_rate', 'spin_axis', 'release_extension',
    'az', 'ax', 'release_pos_x', 'release_pos_z',
    'speed_diff', 'az_diff', 'ax_diff',
    'p_throws', 'stand',
    # New approach angle features
    'VAA', 'HAA', 'vaa_diff',
    # New SSW proxy features
    'ssw_x', 'ssw_z', 'ssw_magnitude',
    # Pitcher baselines
    'avg_fastball_speed', 'avg_fastball_az', 'avg_fastball_ax', 'avg_fastball_vaa',
    # Outcome & count
    'count', 'des_new', 'swing', 'delta_run_exp',
    # Target
    'target',
]

# Step 1: Clean the training data
print("=" * 60)
print("STEP 1: Cleaning training data")
print("=" * 60)
cleaned_train_data = df_clean(all_train_data)
print(f"After cleaning: {cleaned_train_data.shape[0]} rows")

# Step 2: Compute VAA (before pitcher stats lookup — avg_fastball_vaa depends on it)
print("\n" + "=" * 60)
print("STEP 2: Computing VAA")
print("=" * 60)
cleaned_train_data = compute_vaa(cleaned_train_data)
print(f"VAA computed. Null VAA count: {cleaned_train_data['VAA'].null_count()}")

# Step 3: Calculate lookup tables from TRAINING data only (prevent data leakage)
print("\n" + "=" * 60)
print("STEP 3: Calculating statistics from training data")
print("=" * 60)
target_lookup = calculate_target_lookup(cleaned_train_data)
print(f"Target lookup table created with {target_lookup.shape[0]} unique combinations")

pitcher_stats_lookup = calculate_pitcher_stats_lookup(cleaned_train_data)
print(f"Pitcher stats lookup created for {pitcher_stats_lookup.select('pitcher').unique().height} pitchers")

# Step 4: Train Magnus baseline model for SSW proxy (training data only)
print("\n" + "=" * 60)
print("STEP 4: Training Magnus baseline model")
print("=" * 60)
magnus_model_x, magnus_model_z = calculate_magnus_model(cleaned_train_data)

# Step 5: Apply target values to training data
print("\n" + "=" * 60)
print("STEP 5: Applying target values to training data")
print("=" * 60)
train_with_target = apply_target_values(cleaned_train_data, target_lookup)
print(f"Target values applied. Rows with null target: {train_with_target.filter(pl.col('target').is_null()).height}")

# Step 6: Apply feature engineering (HAA, vaa_diff, SSW, differentials)
print("\n" + "=" * 60)
print("STEP 6: Feature engineering on training data")
print("=" * 60)
processed_all_train_data = feature_engineering(
    train_with_target, pitcher_stats_lookup, magnus_model_x, magnus_model_z
)

# Step 7: Select only needed columns
processed_all_train_data = processed_all_train_data.select(
    [c for c in FINAL_COLS if c in processed_all_train_data.columns]
)
print(f"Feature engineering complete. Final shape: {processed_all_train_data.shape}")


In [ ]:
print("\n" + "=" * 60)
print("SAVING LOOKUP TABLES & MODELS")
print("=" * 60)

target_lookup.write_csv('./data/target_lookup.csv')
print("Saved: target_lookup.csv")

pitcher_stats_lookup.write_csv('./data/pitcher_stats_lookup.csv')
print("Saved: pitcher_stats_lookup.csv")

joblib.dump(magnus_model_x, './data/magnus_model_x.joblib')
joblib.dump(magnus_model_z, './data/magnus_model_z.joblib')
print("Saved: magnus_model_x.joblib, magnus_model_z.joblib")


In [ ]:
print("\n" + "=" * 60)
print("SAVING FINAL TRAINING DATA")
print("=" * 60)
processed_all_train_data.write_csv('./data/train_data.csv')
print(f"Saved train_data.csv with {processed_all_train_data.shape[0]} rows and {processed_all_train_data.shape[1]} columns")


SAVING FINAL TRAINING DATA
Saved train_data.csv with 2843813 rows and 128 columns


In [ ]:
!cp "./data/train_data.csv" "/content/drive/MyDrive/stuff_model/data/"

In [ ]:
# ============================================================================
# PROCESS 2025 DATA (NEW PITCHES TO SCORE)
# ============================================================================

# Load 2025 data
print("=" * 60)
print("LOADING 2025 DATA")
print("=" * 60)
data_2025 = pl.read_csv('./data/2025_data.csv')
validate_data(data_2025, "Raw 2025 data")

# Load Magnus models trained on 2021-2024 (do NOT retrain on 2025 data)
magnus_model_x = joblib.load('./data/magnus_model_x.joblib')
magnus_model_z = joblib.load('./data/magnus_model_z.joblib')
print("Loaded Magnus baseline models")

# Step 1: Clean 2025 data
print("\n" + "=" * 60)
print("STEP 1: Cleaning 2025 data")
print("=" * 60)
cleaned_2025 = df_clean(data_2025)
print(f"After cleaning: {cleaned_2025.shape[0]} rows")

# Step 2: Compute VAA (before pitcher stats — avg_fastball_vaa depends on it)
print("\n" + "=" * 60)
print("STEP 2: Computing VAA")
print("=" * 60)
cleaned_2025 = compute_vaa(cleaned_2025)
print(f"VAA computed. Null VAA count: {cleaned_2025['VAA'].null_count()}")

# Step 3: Calculate pitcher stats from 2025 data
# (speed_diff is relative to the pitcher's current year average — standard for Stuff+)
print("\n" + "=" * 60)
print("STEP 3: Calculating pitcher stats for 2025")
print("=" * 60)
pitcher_stats_2025 = calculate_pitcher_stats_lookup(cleaned_2025)
print(f"Pitcher stats lookup created for {pitcher_stats_2025.select('pitcher').unique().height} pitchers in 2025")

# Step 4: Apply feature engineering using 2025 stats + 2021-2024 Magnus models
print("\n" + "=" * 60)
print("STEP 4: Feature engineering on 2025 data")
print("=" * 60)
processed_2025 = feature_engineering(
    cleaned_2025, pitcher_stats_2025, magnus_model_x, magnus_model_z
)

# Step 5: Select only needed columns
processed_2025 = processed_2025.select(
    [c for c in FINAL_COLS if c in processed_2025.columns]
)
print(f"Feature engineering complete. Final shape: {processed_2025.shape}")

# Step 6: Save processed 2025 data
print("\n" + "=" * 60)
print("SAVING PROCESSED 2025 DATA")
print("=" * 60)
processed_2025.write_csv('./data/test_data_2025.csv')
print(f"Saved test_data_2025.csv with {processed_2025.shape[0]} rows and {processed_2025.shape[1]} columns")


In [ ]:
!cp "./data/test_data_2025.csv" "/content/drive/MyDrive/stuff_model/data/"